# 02. Inner-product geometry and numerical stability

![Vectors and stable cosine similarity](../images/02_inner_product_geometry.svg)

Lesson 01 gave every token a feature vector. This notebook answers the next question:
how do we say that two of those vectors are similar, and how do we keep the answer
correct on finite hardware?

**What you will do:** connect dot products to angles, compute norms and cosine
similarity, pool token features, reproduce two floating-point failure modes, and write
explicit zero-norm and tolerance policies.

The [lecture](../lectures/02_inner_product_geometry.md) develops the geometry that the
cells below verify.

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 11
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'numpy={np.__version__}, torch={torch.__version__}, device=cpu')

## 1. Dot products and norms

Two quantities do all the work. The dot product $x^T y=\sum_i x_i y_i$ measures signed
alignment: it grows when coordinates agree in sign and shrinks when they disagree. The
Euclidean norm $\|x\|_2=\sqrt{x^T x}$ measures length alone.

They meet in the identity $x^T y=\|x\|\,\|y\|\cos\theta$, where $\theta$ is the
angle between the vectors. That identity is the reason a raw dot product mixes direction
with magnitude, and it is what the next section divides apart. The vectors below are
perpendicular, so their dot product should be exactly zero.

In [ ]:
x = np.array([3.0, 4.0])
y = np.array([-4.0, 3.0])
dot = x @ y
norm_x, norm_y = np.linalg.norm(x), np.linalg.norm(y)
cosine = dot / (norm_x * norm_y)
assert dot == 0 and norm_x == norm_y == 5
assert np.isclose(cosine, 0.0)
print(f'dot={dot:.1f}, norms=({norm_x:.1f}, {norm_y:.1f}), cosine={cosine:.1f}')

## 2. Cosine similarity removes magnitude

Dividing by both norms cancels the two length factors in the identity above and leaves
only $\cos\theta$. Scaling a vector by a positive number then cannot change its score,
which is exactly the property a direction-only comparison needs.

In code, normalize every vector to unit length first and then take dot products. For
many vectors at once, `X_normalized @ X_normalized.T` computes every pair through
optimized matrix multiplication instead of a Python loop. The diagonal must be one, the
matrix must be symmetric, and opposite vectors must score minus one.

In [ ]:
X = np.array([[1.0, 0.0], [1.0, 1.0], [-2.0, 0.0]])
Xn = X / np.linalg.norm(X, axis=-1, keepdims=True)
S = Xn @ Xn.T
np.testing.assert_allclose(np.diag(S), 1.0, atol=1e-12)
np.testing.assert_allclose(S, S.T, atol=1e-12)
assert np.isclose(S[0, 2], -1.0)
print(np.round(S, 3))

## 3. Mean and masked pooling

A clip is a set of tokens, so comparing clips means reducing that set to one vector
first. For `(B,N,D)` features, `mean(dim=1)` removes only the token axis and leaves batch
and feature axes untouched.

Padded sequences need more care. A plain mean divides by the fixed length `N`, which
would dilute short sequences with padding. A masked mean sums only the valid tokens and
divides by how many there were. Clamping that count prevents division by zero, and a row
with no valid tokens should still be reported explicitly rather than silently returned
as zero.

In [ ]:
tokens = torch.arange(2*4*3, dtype=torch.float32).reshape(2, 4, 3)
mask = torch.tensor([[True, True, False, False], [True, True, True, False]])
weights = mask.unsqueeze(-1).to(tokens.dtype)
counts = weights.sum(dim=1).clamp_min(1.0)
masked_mean = (tokens * weights).sum(dim=1) / counts
assert masked_mean.shape == (2, 3)
torch.testing.assert_close(masked_mean[0], tokens[0, :2].mean(dim=0))
assert mask.any(dim=1).all()
print('masked means:\n', masked_mean)

## 4. Floating-point range and precision

The formulas above assume exact arithmetic. Hardware stores a finite set of values, and
two consequences show up constantly.

First, **range**: `float32` holds magnitudes only up to about $3.4\times10^{38}$, so
squaring a large but finite value can overflow to infinity before the final square root
ever runs. Second, **precision**: `float32` keeps about seven decimal digits, so adding a
tiny number to a much larger one can round away entirely. The cell reproduces both, then
shows that promoting the reduction to `float64` rescues the first case.

In [ ]:
large32 = np.array([1e20, 1e20], dtype=np.float32)
with np.errstate(over='ignore'):
    naive = np.sqrt(np.sum(large32 * large32))
promoted = np.sqrt(np.sum(large32.astype(np.float64) ** 2))
assert np.isinf(naive) and np.isfinite(promoted)

# Adding a tiny value to a much larger float32 value can round away.
rounded = np.float32(1e8) + np.float32(1.0)
assert rounded == np.float32(1e8)
print(f'naive={naive}, promoted={promoted:.3e}, rounded={rounded:.1f}')

## 5. A zero-vector policy

Cosine similarity divides by both norms, and the zero vector has norm zero. The
difficulty is conceptual before it is numerical: direction is a ray from the origin, and
the zero vector has no ray, so it has no angle.

Clamping the denominator with a small `eps` is a computational convention, not a proof
that the angle exists. Under it the zero vector normalizes to zero and therefore scores
zero against everything. Document that choice wherever results depend on it.
`F.normalize` implements the clamped pattern along a chosen axis without building a large
broadcast pair tensor.

In [ ]:
features = torch.tensor([[3.0, 4.0], [0.0, 0.0], [1e-12, 0.0]])
normalized = F.normalize(features, dim=-1, eps=1e-8)
assert torch.isfinite(normalized).all()
torch.testing.assert_close(normalized[0], torch.tensor([0.6, 0.8]))
torch.testing.assert_close(normalized[1], torch.zeros(2))
pairwise = normalized @ normalized.T
print('stable normalized vectors:\n', normalized)
print('pairwise similarities:\n', pairwise)

## 6. Test with relative and absolute tolerances

Because results are rounded, tests need a definition of close enough. The standard rule
is $|a-b| \le atol + rtol\,|b|$, where $a$ is the computed value and $b$ is the
reference.

The two terms cover two regimes. Absolute tolerance sets a small neighborhood around
zero, where relative error can explode. Relative tolerance scales with magnitude, which
is what you want for large references. Choose both from the dtype, the length of the
reduction, and the consequence of the comparison, rather than reusing one number
everywhere.

In [ ]:
a = np.float64(0.1) + np.float64(0.2)
b = np.float64(0.3)
assert a != b
assert np.isclose(a, b, rtol=1e-12, atol=1e-15)

random_features = torch.randn(64, 32)
unit = F.normalize(random_features, dim=-1)
torch.testing.assert_close(torch.linalg.vector_norm(unit, dim=-1), torch.ones(64))
assert torch.isfinite(unit).all()
print('0.1 + 0.2 error:', format(a - b, '.3e'))

## Exercises and final takeaways

**Efficiency.** Normalize reusable candidates once instead of recomputing their norms per
query, and prefer a single matrix product to an `(M,N,D)` broadcast difference.

**Exercises:** (1) Compute the cosine similarity of `(1,2,2)` and `(2,0,1)` by hand, then
verify it here. (2) Add an all-masked sequence and return both the safe mean and an
`is_valid` flag. (3) Compare pairwise cosine results in `float32` and `float64` for
nearly parallel vectors.

**Takeaways:** dot products mix length with alignment; cosine similarity isolates
direction and can therefore reorder a result list; pooling must reduce the intended axis;
stable software states its dtype, epsilon, finite-value, and tolerance policies out
loud.

## Continue learning

[Previous notebook: 01](01_spatiotemporal_tensor_geometry.ipynb) | [Lecture](../lectures/02_inner_product_geometry.md) | [Curriculum](../README.md) | [Next notebook: 03](03_hierarchical_observations.ipynb)